## Identify stable clusters

In [ ]:
# eval "$(conda shell.bash hook)"
# conda init
# conda activate /work/islet_cartography_scrna/scrna_cartography_py_analysis
# python -m ipykernel install --user --name scrna_cartography_py_analysis --display-name "py_analysis"

In [1]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
from pyhere import here      # Reproducible project paths
from pathlib import Path # File system paths

# Single-cell data handling
import anndata as ad         # Core data structure for single-cell data
import scanpy as sc          # Analysis and visualization of single-cell data

# Data visualization
import seaborn as sns        # Statistical data visualization
import matplotlib.pyplot as plt  # Plotting interface
import matplotlib            # Base matplotlib functionality
from matplotlib.backends.backend_pdf import PdfPages  # Save plots to multi-page PDFs
import tol_colors            # colors

# Parallel processing
from joblib import Parallel, delayed, parallel_backend

# Jaccard score
from sklearn.metrics import jaccard_score

# Dataframes
import pandas as pd
import numpy as np

# Miscellaneous utilities
import warnings              # Suppress or manage warnings

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import my_anndata as ma                    # Custom AnnData utilities

Parameters

In [2]:
base_dir = str(here('data/annotate'))
plot_dir = os.path.join(base_dir, 'plot') 
objects_dir = os.path.join(base_dir, 'objects') 
files_dir = os.path.join(base_dir, 'files') 

anndata_dir = str(here('data/anndata/'))

# Seed
seed = 1000

Functions

In [3]:
def compute_jaccard(original_labels: pd.Categorical, bootstrap_labels: pd.Categorical) -> list:
    """
    Compute Jaccard index between original clustering and bootstrap clustering.

    Parameters
    ----------
    original_labels : pandas.Categorical
        Cluster labels from the original clustering (e.g., adata.obs['clusters']).
    bootstrap_labels : pandas.Categorical
        Cluster labels from the bootstrap clustering.

    Returns
    -------
    list
        Jaccard similarity scores for each original cluster.

    Notes
    -----
    For each cluster in the original labels:
        1. Create a binary mask of membership for that cluster.
        2. Compare it to every cluster in the bootstrap labels using Jaccard similarity.
        3. Keep the largest Jaccard score (i.e., the most similar bootstrap cluster).
        4. Append this best score to the results list.
    """

    # Get unique clusters from original dataset
    unique_clusters = np.unique(original_labels)
    scores = []

    # For each original cluster, calculate Jaccard score with all bootstrap clusters
    for cluster in unique_clusters:
        # Binary mask for the original cluster (1 = in cluster, 0 = not in cluster)
        orig_mask = (original_labels == cluster).astype(int)
        best_score = 0

        # Compare to all bootstrap clusters
        for b_cluster in np.unique(bootstrap_labels):
            # Binary mask for the bootstrap cluster
            boot_mask = (bootstrap_labels == b_cluster).astype(int)
            # Compute Jaccard similarity
            score = jaccard_score(orig_mask, boot_mask)
            # Keep the best (highest) score for this original cluster
            if score > best_score:
                best_score = score

        # Append best score for this cluster
        scores.append(best_score)

    return scores

Import data

In [4]:
# adata 
adata = ad.read_h5ad(os.path.join(anndata_dir, "AG_combined.h5ad"))

Stable clusters

In [ ]:
# Find neighbors
with parallel_backend("threading", n_jobs=60): 
    sc.pp.neighbors(adata, use_rep='X_latent_1') 
    
# Compute umap
sc.tl.umap(adata)

In [ ]:
adata.write(os.path.join(anndata_dir, "AG_combined.h5ad"))

In [16]:
n_bootstrap = 100
resolutions = np.round(np.arange(0.05, 0.75, 0.05), 2)
min_cells = 10

# Dictionary to store stability results
stability_results = {}

# For each res do bootstrapping for cluster stability
for res in resolutions:
    print(f'finding clusters with resolution: {res}')

    # Find "original" clusters
    sc.tl.leiden(adata, flavor='igraph', resolution = res, random_state= seed, n_iterations= 2, key_added=f'leiden_{res}')
    
    # Get labels
    original_labels = adata.obs[f'leiden_{res}'] 

    # List for jaccard scores:
    jaccard_scores_bootstrap = []
    
    for boot in range(n_bootstrap):
        # Subset AnnData object to 10 % of cells
        sample_size = int(adata.n_obs * 0.1)  # 10% of cells
        sample_idx = np.random.choice(adata.n_obs, sample_size, replace=False) # to avoid repeated barcodes with replace = False
        adata_subset = adata[sample_idx, :].copy()

        # Find neighbors and clusters
        with parallel_backend("threading", n_jobs=60): 
            sc.pp.neighbors(adata_subset, use_rep='X_latent_1') 
        
        sc.tl.leiden(adata_subset, flavor='igraph', resolution = res, random_state= seed, n_iterations= 2, key_added='bootstrap_clusters')

        # Get cluster labels, make sure index-aligned original labels for this subset
        original_labels_subset = original_labels.iloc[sample_idx].copy()
        bootstrap_labels = adata_subset.obs['bootstrap_clusters'].copy()

        # Remove small clusters from original labels
        cluster_sizes = original_labels_subset.value_counts()
        valid_clusters = cluster_sizes[cluster_sizes >= min_cells].index

        # remove barcodes that were in small clusters from original and bootstrap labels
        mask = np.isin(original_labels_subset, valid_clusters)
        original_labels_filtered = original_labels_subset[mask].values
        bootstrap_labels_filtered = bootstrap_labels[mask].values

        
        # Calculated jaccard index between original dataset and bootstrap
        if len(original_labels_filtered) > 0 and len(np.unique(original_labels_filtered)) > 10: # avoid empty input and too few clusters
            scores = compute_jaccard(original_labels_filtered, bootstrap_labels_filtered)
            # Add jaccard scores to list
            jaccard_scores_bootstrap.extend(scores)

    # Add everything to a dictionary 
    stability_results[res] = jaccard_scores_bootstrap

# Create dataframe
data = []
for res, scores in stability_results.items():
    for score in scores:
        data.append({'resolution': res, 'jaccard': score})

results = pd.DataFrame(data)    

finding clusters with resolution: 0.05
finding clusters with resolution: 0.1
finding clusters with resolution: 0.15
finding clusters with resolution: 0.2
finding clusters with resolution: 0.25
finding clusters with resolution: 0.3


FileNotFoundError: [Errno 2] No such file or directory

Save results

In [ ]:
# The leiden clusters
clustering = adata.obs.loc[:,adata.obs.columns.str.startswith('leiden_')].copy()
clustering.to_csv(os.path.join(files_dir, 'leiden_clusterings.csv'), index_label='barcode')

# The results
results.to_csv(os.path.join(files_dir, 'clustering_stability_results.csv'), index_label='cluster')

In [ ]:
results = pd.read_csv(os.path.join(files_dir, 'clustering_stability_results.csv'), index_col=0)

In [ ]:
def set_my_theme():
    sns.set_style("white")  # similar to theme_classic background

    matplotlib.rcParams['pdf.fonttype'] = 42
    matplotlib.rcParams['ps.fonttype'] = 42

    plt.rcParams.update({
        # text + labels
        "axes.titlesize": 8,
        "axes.titleweight": "normal",
        "axes.labelsize": 7,
        "xtick.labelsize": 6,
        "ytick.labelsize": 6,
        "legend.fontsize": 4,
        "legend.title_fontsize": 4,

        # line & tick colors
        "axes.edgecolor": "black",
        "xtick.color": "black",
        "ytick.color": "black",
        "text.color": "black",

        # spine & tick thickness
        "axes.linewidth": 0.7,
        "xtick.major.width": 0.7,
        "ytick.major.width": 0.7,

        # IMPORTANT: make ticks long & visible
        "xtick.major.size": 4,
        "ytick.major.size": 4,
        "xtick.bottom": True,
        "ytick.left": True,

        # background
        "axes.facecolor": "white",
        "figure.facecolor": "white",
        "savefig.facecolor": "white",
    })

In [ ]:
results['resolution'].unique()

In [ ]:
set_my_theme()
plt.figure(figsize=(7,5))

# Boxplot (hide outlier markers so points look cleaner)
sns.boxplot(
    x='resolution',
    y='jaccard',
    data=results,
    fliersize=0, 
    fill=False, 
    color="black"
)

# Overlay individual points
sns.stripplot(
    x='resolution',
    y='jaccard',
    data=results,
    alpha=0.5, size=1,
    jitter=True, 
    color="grey"
)

# Horizontal reference line
plt.axhline(y=0.9, linewidth=2, color='red', ls=':')

plt.xlabel('Resolution')
plt.ylabel('Jaccard Index')
plt.title('Cluster Stability Across Resolutions')
plt.tight_layout()
plt.show()

# so it seems all clusters are stable

In [ ]:
stability_summary = results.groupby('resolution')['jaccard'].median()
best_resolution = stability_summary.idxmax()
max_median_jaccard = stability_summary.max()
stability_summary
print(f"The resolution with the highest median Jaccard is {best_resolution}, with a median of {max_median_jaccard:.4f}")
print(stability_summary)

In [ ]:
clustering = pd.read_csv(os.path.join(files_dir, 'leiden_clusterings.csv'), index_col=0)
clustering

In [ ]:
import networkx as nx

In [ ]:
# Ensure clusters are strings
df = clustering.astype(str)

# Sort columns by resolution number
res_cols = sorted([c for c in df.columns if c.startswith('leiden')],
                  key=lambda x: float(x.split("_")[1]))

# Build edges with counts
edges = []
for i in range(len(res_cols)-1):
    res1, res2 = res_cols[i], res_cols[i+1]
    temp = df[[res1, res2]]
    edge_counts = temp.groupby([res1, res2]).size().reset_index(name='count')
    for _, row in edge_counts.iterrows():
        edges.append((f"{res1}_{row[res1]}", f"{res2}_{row[res2]}", row['count']))

Nodes: Each cluster at each resolution.

Edges: Flow of cells from one cluster to the next resolution.

Node size: Number of cells in that cluster.

Edge width: Number of cells moving from cluster A at res N → cluster B at res N+1.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import os

# Load your dataframe
df = pd.read_csv(os.path.join(files_dir, 'leiden_clusterings.csv'), index_col=0)

# Build nodes
node_labels = []
node_map = {}
index = 0
res_columns = df.columns[1:]
for res in res_columns:
    for clust in df[res].unique():
        node_map[(res, clust)] = index
        node_labels.append(str(clust))
        index += 1

# Build links
source, target, value = [], [], []
for i in range(len(res_columns)-1):
    res1, res2 = res_columns[i], res_columns[i+1]
    grouped = df.groupby([res1, res2]).size().reset_index(name='count')
    for _, row in grouped.iterrows():
        source.append(node_map[(res1, row[res1])])
        target.append(node_map[(res2, row[res2])])
        value.append(row['count'])

# Assign x positions (normalized 0-1) for layers
x_pos = []
for n in node_map:
    res, clust = n
    x_pos.append(res_columns.tolist().index(res) / (len(res_columns)-1))

# Sankey diagram
fig = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(
        pad=15,
        thickness=20,
        label=node_labels,
        x=x_pos,
        y=None  # Plotly auto-arranges y
    ),
    link=dict(
        source=source,
        target=target,
        value=value
    )
))

# Update layout: wider and add resolution annotations
fig.update_layout(
    width=1200,  # make figure wider
    height=600,
    title_text="Cluster Tree",
    font_size=8,
)

col_label = list(map( lambda x: x.replace( 'leiden_', ''), res_columns))

# Optional: add annotations for resolutions
annotations = []
for i, res in enumerate(col_label):
    annotations.append(dict(
        x=i/(len(col_label)-1),
        y=1.05,  # above the plot
        xref="paper",
        yref="paper",
        text=res,
        showarrow=False,
        font=dict(size=8)
    ))

fig.update_layout(annotations=annotations)
fig.show()


In [ ]:
list(col_label)

In [ ]:
node_labels